# Random Forest vs Best Individual Model Comparison

This notebook compares the Random Forest Classifier against the best individual model from Part B (SVM).

**Analysis includes:**
- Side-by-side metrics comparison table
- Train vs Test performance analysis
- Overfitting and generalization discussion
- Variance reduction via bagging and feature subsampling
- Stability across classes analysis
- Final conclusion on generalization performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
RANDOM_STATE = 42
sns.set_style('whitegrid')

print("✓ All libraries imported successfully!")

In [ ]:
# Record library versions for reproducibility
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
try:
    print(f"Random State: {RANDOM_STATE}")
except NameError:
    print("Random State: Not defined yet")
print("="*60)

## 1. Load and Prepare Data

Using the same data preparation as previous notebooks for consistency.

In [ ]:
# Load dataset
df = pd.read_csv('my_data .csv')

# Split features and target
X = df.drop(columns=['placed'])
y = df['placed']

# Identify feature types
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(num_features)} numerical, {len(cat_features)} categorical")

# Stratified train/test split (25% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 2. Define Preprocessing Pipeline

Same preprocessing pipeline used in previous notebooks.

In [ ]:
# Define transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

print("✓ Preprocessing pipeline created")

## 3. Train Both Models

Training the best individual model (SVM) and Random Forest for comparison.

In [ ]:
# Model A: SVM (Best individual model from Part B)
print("Training SVM (Best Individual Model)...")
svm_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE, probability=True))
])
svm_model.fit(X_train, y_train)
print("✓ SVM trained")

# Model B: Random Forest
print("\nTraining Random Forest...")
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)
print("✓ Random Forest trained")

print("\n" + "="*60)
print("Both models trained successfully!")
print("="*60)

## 4. Evaluate Models on Train and Test Sets

Computing metrics on both train and test sets to analyze overfitting.

In [ ]:
def compute_metrics(model, X, y, dataset_name):
    """Compute all metrics for a given dataset"""
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    return {
        'Dataset': dataset_name,
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1-Score': f1_score(y, y_pred),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }

# Evaluate SVM
print("Evaluating SVM...")
svm_train_metrics = compute_metrics(svm_model, X_train, y_train, 'Train')
svm_test_metrics = compute_metrics(svm_model, X_test, y_test, 'Test')

# Evaluate Random Forest
print("Evaluating Random Forest...")
rf_train_metrics = compute_metrics(rf_model, X_train, y_train, 'Train')
rf_test_metrics = compute_metrics(rf_model, X_test, y_test, 'Test')

print("✓ Evaluation complete")

## 5. Comparison Table: Test Set Performance

Side-by-side comparison of key metrics on the test set.

In [ ]:
# Create comparison table for test set
comparison_data = {
    'Model': ['SVM (Best Individual)', 'Random Forest'],
    'Accuracy': [svm_test_metrics['Accuracy'], rf_test_metrics['Accuracy']],
    'Precision': [svm_test_metrics['Precision'], rf_test_metrics['Precision']],
    'Recall': [svm_test_metrics['Recall'], rf_test_metrics['Recall']],
    'F1-Score': [svm_test_metrics['F1-Score'], rf_test_metrics['F1-Score']]
}

comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
print("\n" + "="*80)
print("TEST SET PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Determine winner for each metric
print("\n📊 Metric Winners:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    svm_val = comparison_df[comparison_df['Model'] == 'SVM (Best Individual)'][metric].values[0]
    rf_val = comparison_df[comparison_df['Model'] == 'Random Forest'][metric].values[0]
    winner = 'Random Forest' if rf_val > svm_val else 'SVM' if svm_val > rf_val else 'Tie'
    print(f"   {metric:12s}: {winner} (SVM: {svm_val:.4f}, RF: {rf_val:.4f})")

## 6. Train vs Test Performance Analysis

Analyzing the gap between train and test performance to identify overfitting.

In [ ]:
# Create train vs test comparison
train_test_data = {
    'Model': ['SVM - Train', 'SVM - Test', 'RF - Train', 'RF - Test'],
    'Accuracy': [
        svm_train_metrics['Accuracy'], svm_test_metrics['Accuracy'],
        rf_train_metrics['Accuracy'], rf_test_metrics['Accuracy']
    ],
    'Precision': [
        svm_train_metrics['Precision'], svm_test_metrics['Precision'],
        rf_train_metrics['Precision'], rf_test_metrics['Precision']
    ],
    'Recall': [
        svm_train_metrics['Recall'], svm_test_metrics['Recall'],
        rf_train_metrics['Recall'], rf_test_metrics['Recall']
    ],
    'F1-Score': [
        svm_train_metrics['F1-Score'], svm_test_metrics['F1-Score'],
        rf_train_metrics['F1-Score'], rf_test_metrics['F1-Score']
    ]
}

train_test_df = pd.DataFrame(train_test_data)

print("\n" + "="*80)
print("TRAIN vs TEST PERFORMANCE")
print("="*80)
print(train_test_df.to_string(index=False))
print("="*80)

# Calculate performance gaps
print("\n📉 Performance Gaps (Train - Test):")
print("\nSVM:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    gap = svm_train_metrics[metric] - svm_test_metrics[metric]
    print(f"   {metric:12s}: {gap:+.4f} ({gap*100:+.2f}%)")

print("\nRandom Forest:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    gap = rf_train_metrics[metric] - rf_test_metrics[metric]
    print(f"   {metric:12s}: {gap:+.4f} ({gap*100:+.2f}%)")

# Calculate average gap
svm_avg_gap = np.mean([
    svm_train_metrics['Accuracy'] - svm_test_metrics['Accuracy'],
    svm_train_metrics['F1-Score'] - svm_test_metrics['F1-Score']
])
rf_avg_gap = np.mean([
    rf_train_metrics['Accuracy'] - rf_test_metrics['Accuracy'],
    rf_train_metrics['F1-Score'] - rf_test_metrics['F1-Score']
])

print(f"\n📊 Average Performance Gap:")
print(f"   SVM:           {svm_avg_gap:+.4f} ({svm_avg_gap*100:+.2f}%)")
print(f"   Random Forest: {rf_avg_gap:+.4f} ({rf_avg_gap*100:+.2f}%)")
print(f"\n   → {'Random Forest' if rf_avg_gap < svm_avg_gap else 'SVM'} shows better generalization (smaller gap)")

## 7. Visualization: Performance Comparison

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Test Set Metrics Comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.35

svm_values = [comparison_df[comparison_df['Model'] == 'SVM (Best Individual)'][m].values[0] for m in metrics_to_plot]
rf_values = [comparison_df[comparison_df['Model'] == 'Random Forest'][m].values[0] for m in metrics_to_plot]

axes[0].bar(x - width/2, svm_values, width, label='SVM', color='#2E86AB', alpha=0.8)
axes[0].bar(x + width/2, rf_values, width, label='Random Forest', color='#2E7D32', alpha=0.8)
axes[0].set_xlabel('Metrics', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('Test Set Performance Comparison', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for i, (s, r) in enumerate(zip(svm_values, rf_values)):
    axes[0].text(i - width/2, s + 0.02, f'{s:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].text(i + width/2, r + 0.02, f'{r:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: Train vs Test Gap Comparison
gap_metrics = ['Accuracy', 'F1-Score']
svm_gaps = [svm_train_metrics[m] - svm_test_metrics[m] for m in gap_metrics]
rf_gaps = [rf_train_metrics[m] - rf_test_metrics[m] for m in gap_metrics]

x2 = np.arange(len(gap_metrics))
axes[1].bar(x2 - width/2, svm_gaps, width, label='SVM', color='#2E86AB', alpha=0.8)
axes[1].bar(x2 + width/2, rf_gaps, width, label='Random Forest', color='#2E7D32', alpha=0.8)
axes[1].set_xlabel('Metrics', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Performance Gap (Train - Test)', fontsize=12, fontweight='bold')
axes[1].set_title('Overfitting Analysis: Train-Test Gap', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(gap_metrics)
axes[1].legend(fontsize=11)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for i, (s, r) in enumerate(zip(svm_gaps, rf_gaps)):
    axes[1].text(i - width/2, s + 0.005, f'{s:.3f}', ha='center', va='bottom', fontsize=9)
    axes[1].text(i + width/2, r + 0.005, f'{r:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('rf_vs_svm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Comparison plot saved as 'rf_vs_svm_comparison.png'")

## 8. Class Stability Analysis

Analyzing false negatives vs false positives to assess stability across classes.

In [ ]:
# Analyze confusion matrix patterns
print("\n" + "="*80)
print("CLASS STABILITY ANALYSIS (Test Set)")
print("="*80)

print("\nSVM Confusion Matrix:")
print(f"   True Negatives:  {svm_test_metrics['TN']:4d}")
print(f"   False Positives: {svm_test_metrics['FP']:4d}  ← Incorrectly predicted as Placed")
print(f"   False Negatives: {svm_test_metrics['FN']:4d}  ← Incorrectly predicted as Not Placed")
print(f"   True Positives:  {svm_test_metrics['TP']:4d}")
print(f"   Total Errors:    {svm_test_metrics['FP'] + svm_test_metrics['FN']:4d}")

print("\nRandom Forest Confusion Matrix:")
print(f"   True Negatives:  {rf_test_metrics['TN']:4d}")
print(f"   False Positives: {rf_test_metrics['FP']:4d}  ← Incorrectly predicted as Placed")
print(f"   False Negatives: {rf_test_metrics['FN']:4d}  ← Incorrectly predicted as Not Placed")
print(f"   True Positives:  {rf_test_metrics['TP']:4d}")
print(f"   Total Errors:    {rf_test_metrics['FP'] + rf_test_metrics['FN']:4d}")

# Calculate error balance
svm_error_balance = abs(svm_test_metrics['FP'] - svm_test_metrics['FN'])
rf_error_balance = abs(rf_test_metrics['FP'] - rf_test_metrics['FN'])

print("\n📊 Error Balance (|FP - FN|):")
print(f"   SVM:           {svm_error_balance} (lower is more balanced)")
print(f"   Random Forest: {rf_error_balance} (lower is more balanced)")
print(f"\n   → {'Random Forest' if rf_error_balance < svm_error_balance else 'SVM'} shows more balanced errors across classes")

# Calculate error reduction
total_errors_svm = svm_test_metrics['FP'] + svm_test_metrics['FN']
total_errors_rf = rf_test_metrics['FP'] + rf_test_metrics['FN']
error_reduction = total_errors_svm - total_errors_rf
error_reduction_pct = (error_reduction / total_errors_svm * 100) if total_errors_svm > 0 else 0

print(f"\n📉 Total Error Reduction:")
print(f"   Random Forest reduces errors by {error_reduction} ({error_reduction_pct:+.1f}%) compared to SVM")

## 9. Overfitting & Generalization Discussion

In [ ]:
print("\n" + "="*80)
print("OVERFITTING & GENERALIZATION ANALYSIS")
print("="*80)

discussion = f"""
📝 DISCUSSION:

1. TRAIN-TEST PERFORMANCE GAP:
   The Random Forest model exhibits a train-test performance gap of {rf_avg_gap:.4f} 
   ({rf_avg_gap*100:.2f}%), compared to SVM's gap of {svm_avg_gap:.4f} ({svm_avg_gap*100:.2f}%).
   {'A smaller gap in Random Forest indicates better generalization and less overfitting.' if rf_avg_gap < svm_avg_gap else 'SVM shows a smaller gap, indicating better generalization in this case.'}
   The {'Random Forest' if rf_avg_gap < svm_avg_gap else 'SVM'} model demonstrates superior ability to maintain 
   consistent performance on unseen data.

2. VARIANCE REDUCTION VIA BAGGING:
   Random Forest leverages bootstrap aggregating (bagging) by training {rf_model.named_steps['classifier'].n_estimators} 
   decision trees on different random subsets of the training data. This ensemble approach 
   reduces variance by averaging predictions across multiple trees, each capturing different 
   patterns in the data. Additionally, feature subsampling (random feature selection at each 
   split) further decorrelates the trees, preventing any single feature or pattern from 
   dominating the model and reducing the risk of overfitting to noise.

3. FEATURE SUBSAMPLING BENEFITS:
   By randomly selecting a subset of features at each split, Random Forest ensures that 
   individual trees are less likely to overfit to specific feature combinations. This 
   mechanism is particularly valuable in datasets with many features or correlated predictors, 
   as it forces the model to explore diverse decision boundaries and improves robustness.

4. STABILITY ACROSS CLASSES:
   Examining the confusion matrices, Random Forest produces {rf_test_metrics['FP']} false positives 
   and {rf_test_metrics['FN']} false negatives (error balance: {rf_error_balance}), while SVM produces 
   {svm_test_metrics['FP']} false positives and {svm_test_metrics['FN']} false negatives (error balance: {svm_error_balance}).
   {'Random Forest demonstrates more balanced error distribution across classes, indicating stable performance for both positive and negative predictions.' if rf_error_balance < svm_error_balance else 'SVM shows more balanced errors, suggesting better stability across classes.'}
   This balance is crucial for real-world applications where misclassification costs may differ 
   between classes.

5. TEST SET PERFORMANCE:
   On the test set, Random Forest achieves an F1-score of {rf_test_metrics['F1-Score']:.4f} compared 
   to SVM's {svm_test_metrics['F1-Score']:.4f}, with accuracy of {rf_test_metrics['Accuracy']:.4f} vs 
   {svm_test_metrics['Accuracy']:.4f}. {'Random Forest outperforms SVM on the test set.' if rf_test_metrics['F1-Score'] > svm_test_metrics['F1-Score'] else 'SVM outperforms Random Forest on the test set.'}
"""

print(discussion)
print("="*80)

## 10. Final Conclusion

In [ ]:
# Determine overall winner based on multiple factors
rf_better_test = rf_test_metrics['F1-Score'] > svm_test_metrics['F1-Score']
rf_better_gap = rf_avg_gap < svm_avg_gap
rf_better_balance = rf_error_balance < svm_error_balance

rf_score = sum([rf_better_test, rf_better_gap, rf_better_balance])
svm_score = 3 - rf_score

print("\n" + "="*80)
print("FINAL CONCLUSION")
print("="*80)

if rf_score >= 2:
    conclusion = f"""
🏆 VERDICT: Random Forest generalizes better on this dataset.

Random Forest demonstrates superior generalization compared to SVM through {'a smaller train-test ' if rf_better_gap else ''}
{'performance gap, ' if rf_better_gap else ''}{'better test set performance, ' if rf_better_test else ''}
{'and more balanced error distribution across classes' if rf_better_balance else ''}—the ensemble's 
variance reduction via bagging and feature subsampling effectively mitigates overfitting while 
maintaining robust predictive accuracy on unseen data.
    """
else:
    conclusion = f"""
🏆 VERDICT: SVM generalizes better on this dataset.

Despite Random Forest's ensemble advantages, SVM demonstrates superior generalization through 
{'a smaller train-test performance gap, ' if not rf_better_gap else ''}{'better test set performance, ' if not rf_better_test else ''}
{'and more balanced error distribution' if not rf_better_balance else ''}—the RBF kernel's ability to capture 
complex decision boundaries while maintaining regularization proves more effective for this 
particular dataset's characteristics.
    """

print(conclusion.strip())
print("\n" + "="*80)

# Summary scores
print("\n📊 Generalization Score (out of 3):")
print(f"   Random Forest: {rf_score}/3")
print(f"   SVM:           {svm_score}/3")
print("\n   Criteria: (1) Better test F1-score, (2) Smaller train-test gap, (3) More balanced errors")

## 11. Summary: Return Comparison Table and Commentary

In [ ]:
# Create final results dictionary
results = {
    'comparison_table': comparison_df,
    'train_test_analysis': train_test_df,
    'performance_gaps': {
        'SVM': {
            'accuracy_gap': svm_train_metrics['Accuracy'] - svm_test_metrics['Accuracy'],
            'f1_gap': svm_train_metrics['F1-Score'] - svm_test_metrics['F1-Score'],
            'average_gap': svm_avg_gap
        },
        'Random_Forest': {
            'accuracy_gap': rf_train_metrics['Accuracy'] - rf_test_metrics['Accuracy'],
            'f1_gap': rf_train_metrics['F1-Score'] - rf_test_metrics['F1-Score'],
            'average_gap': rf_avg_gap
        }
    },
    'class_stability': {
        'SVM': {
            'FP': int(svm_test_metrics['FP']),
            'FN': int(svm_test_metrics['FN']),
            'error_balance': int(svm_error_balance),
            'total_errors': int(total_errors_svm)
        },
        'Random_Forest': {
            'FP': int(rf_test_metrics['FP']),
            'FN': int(rf_test_metrics['FN']),
            'error_balance': int(rf_error_balance),
            'total_errors': int(total_errors_rf)
        }
    },
    'winner': 'Random Forest' if rf_score >= 2 else 'SVM',
    'generalization_scores': {
        'Random_Forest': int(rf_score),
        'SVM': int(svm_score)
    }
}

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print("\n✅ Comparison table created")
print("✅ Overfitting analysis completed")
print("✅ Generalization discussion provided")
print("✅ Class stability assessed")
print("✅ Final conclusion delivered")
print("\n📊 Comparison plot saved: rf_vs_svm_comparison.png")
print("="*80)

# Display final comparison table one more time
print("\n📋 FINAL COMPARISON TABLE (Test Set):")
print(comparison_df.to_string(index=False))

results

## Summary

This notebook has successfully:

1. ✅ **Loaded and prepared data** using the same approach as previous notebooks
2. ✅ **Trained both models** (SVM and Random Forest) with identical preprocessing
3. ✅ **Created comparison table** showing accuracy, precision, recall, and F1-score
4. ✅ **Analyzed overfitting** by comparing train vs test performance gaps
5. ✅ **Discussed variance reduction** via bagging and feature subsampling
6. ✅ **Assessed class stability** by examining false negatives vs false positives
7. ✅ **Provided final conclusion** on which model generalizes better
8. ✅ **Generated visualizations** comparing both models

### Connection to Previous Notebooks

This notebook connects to:
- **`model_evaluation_comparison.ipynb`**: Uses the same SVM model (best individual model from Part B)
- **`random_forest_model.ipynb`**: Compares against the Random Forest model trained there
- **Same preprocessing pipeline**: Ensures fair comparison across all models
- **Same train/test split**: Uses identical random_state=42 for reproducibility